# 03. From particles to detector objects

Compare MadGraph-only and full-pipeline samples with the same beams, process,
model, and generator cuts. Independent samples need not contain the same events
or use the same seed.

Compare both normalized shapes and accepted counts per generated event.
The difference combines radiation and detector effects; it does not isolate them.

Copy this notebook into `work`, select **Python (hep)**, and run cells from the
top. Choose sample IDs from the table below before plotting.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
sample_ids = []  # [parton_sample_id, detector_sample_id]
observable = "mll"
if sample_ids:
    samples = [get_sample(key) for key in sample_ids]
    frames = [s.load() for s in samples]
    values = pd.concat([f[observable] for f in frames]).dropna()
    bins = np.linspace(values.min()-0.1, values.max()+0.1,41)
    fig, ax = plt.subplots(figsize=(7,4))
    for s, f in zip(samples,frames):
        w = weights(f)
        if w.sum() <= 0:
            raise ValueError("This shape comparison requires positive total weight.")
        ax.hist(f[observable], bins=bins, weights=w/w.sum(),histtype="step",label=s.label)
        print(s.label, "stored pair rows / generated:", len(f), "/", s.generated_events)
        print(s.config)
    ax.set_xlabel(observable); ax.set_ylabel("Fraction of stored sample"); ax.legend(); plt.show()

## Questions and submission
1. Repeat for mll, ptll, and leading_lepton_pt.
2. Explain why pair recoil is no longer constrained to zero after radiation.
3. List at least two mechanisms that can move or broaden the observed spectrum.
4. What intermediate information would you need to separate those mechanisms?

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.